# 02. Модельді баптау (Fine-tuning)

**Алдыңғы қадам:** `01_data_preparation.ipynb` → 45k+ жазба дайындалды  
**Бұл ноутбук:**
1. Деректерді Google Drive-тан жүктейді
2. `ai-forever/ruBert-base` (178M параметр) моделін баптайды
3. Валидация / тест метрикаларын есептейді
4. ONNX + INT8 квантизация жасайды
5. `model.onnx` + `tokenizer/` → FastAPI-ге деплой үшін сақтайды

**Орта:** Google Colab T4 GPU  
**Болжамды уақыт:** ~60–80 мин (4 эпоха, 31k train сэмпл)

---
> **Бастамас бұрын:** Runtime → Change runtime type → **T4 GPU** таңдаңыз

In [ ]:
# 1. Тәуелділіктерді орнату
# transformers>=4.43 қажет (EncoderDecoderCache үшін)
!pip install -q -U transformers datasets accelerate scikit-learn
!pip install -q onnx onnxruntime optimum[onnxruntime]
!pip install -q matplotlib seaborn tqdm numpy

import transformers
print(f'transformers версиясы: {transformers.__version__}')
print('Орнату аяқталды.')

In [ ]:
# 2. Google Drive қосу
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/disinformation_project'
os.makedirs(f'{SAVE_DIR}/models', exist_ok=True)
print(f'SAVE_DIR: {SAVE_DIR}')

In [ ]:
# 3. Конфигурация
import torch

# ── Модель ──────────────────────────────────────────────────────────────
MODEL_NAME = 'ai-forever/ruBert-base'
# Баламалар (ai-forever қол жетімді болмаса):
#   'DeepPavlov/rubert-base-cased'  — сапасы ұқсас
#   'cointegrated/rubert-tiny2'     — 29M, жылдамырақ, сапасы төменірек

# ── Оқыту гиперпараметрлер ──────────────────────────────────────────────
MAX_LENGTH  = 256   # app/config.py-мен сәйкес
BATCH_SIZE  = 16    # T4 (16GB) үшін оңтайлы; OOM болса 8-ге азайтыңыз
EPOCHS      = 4     # 3–5 эпоха жеткілікті
LR          = 2e-5
WARMUP      = 0.1   # warmup_ratio
WD          = 0.01  # weight decay
GRAD_ACCUM  = 2     # eff. batch = BATCH_SIZE × GRAD_ACCUM = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}  ({gpu_mem:.1f} GB)')
else:
    print('ЕСКЕРТУ: GPU табылмады — CPU-де оқыту өте баяу болады!')

print(f'Модель:        {MODEL_NAME}')
print(f'MAX_LENGTH:    {MAX_LENGTH}')
print(f'Батч (eff.):   {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}')
print(f'Эпохалар:      {EPOCHS}')
print(f'Learning rate: {LR}')

## 1. Деректерді жүктеу

In [ ]:
# 4. Деректерді жүктеу (01_data_preparation.ipynb-тен сақталған)
from datasets import load_from_disk

dataset_path = f'{SAVE_DIR}/data/disinfo_dataset'
dataset = load_from_disk(dataset_path)

print(dataset)
print()

# Класс балансын тексеру
for split_name, split in dataset.items():
    n1 = sum(1 for x in split['label'] if x == 1)
    n0 = len(split) - n1
    print(f'{split_name:<12s} {len(split):>6d} жазба  '
          f'(label=0: {n0}, label=1: {n1}, '
          f'дезинформация: {n1/len(split):.1%})')

In [ ]:
# 5. Мысал мәтіндерді тексеру
import random
random.seed(42)

print('=== 3 СЕНІМДІ мысал (label=0) ===')
idxs0 = [i for i, l in enumerate(dataset['train']['label']) if l == 0]
for i in random.sample(idxs0, 3):
    print(f'  {dataset["train"]["text"][i][:120]}...')
    print()

print('=== 3 ДЕЗИНФОРМАЦИЯ мысал (label=1) ===')
idxs1 = [i for i, l in enumerate(dataset['train']['label']) if l == 1]
for i in random.sample(idxs1, 3):
    print(f'  {dataset["train"]["text"][i][:120]}...')
    print()

## 2. Токенизация

In [ ]:
# 6. Токенайзер жүктеу
from transformers import AutoTokenizer

print(f'{MODEL_NAME} токенайзері жүктелуде...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Токен ұзындығын тексеру
import numpy as np
sample_texts = dataset['train']['text'][:500]
lengths = [len(tokenizer.encode(t)) for t in sample_texts]
print(f'Токен ұзындығы (500 сэмпл):')
print(f'  Медиана: {np.median(lengths):.0f}')
print(f'  90-пайыздық: {np.percentile(lengths, 90):.0f}')
print(f'  Макс: {max(lengths)}')
print(f'  MAX_LENGTH={MAX_LENGTH}-ден ұзын: {sum(l > MAX_LENGTH for l in lengths)} / {len(lengths)}')

In [ ]:
# 7. Деректерді токенизациялау
def tokenize_fn(examples):
    return tokenizer(
        examples['text'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length',
    )

print('Токенизация жүріп жатыр...')
tokenized = dataset.map(
    tokenize_fn,
    batched=True,
    batch_size=512,
    remove_columns=['text'],
    desc='Токенизация',
)
tokenized.set_format('torch')

print(tokenized)
print(f'\nОриентировочный размер (train): {len(tokenized["train"])} × {MAX_LENGTH} × 3 int32 = '
      f'{len(tokenized["train"]) * MAX_LENGTH * 3 * 4 / 1e6:.0f} MB')

## 3. Модель жүктеу және оқыту

In [ ]:
# 8. Модельді жүктеу
from transformers import AutoModelForSequenceClassification

print(f'{MODEL_NAME} жүктелуде...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'reliable', 1: 'disinformation'},
    label2id={'reliable': 0, 'disinformation': 1},
).to(device)

total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Барлық параметрлер: {total:,}')
print(f'Оқытылатын:         {trainable:,}')

In [ ]:
# 9. Метрика функциясы
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    preds  = np.argmax(eval_pred.predictions, axis=-1)
    labels = eval_pred.label_ids
    return {
        'accuracy':  accuracy_score(labels, preds),
        'f1':        f1_score(labels, preds, average='binary'),
        'precision': precision_score(labels, preds, average='binary', zero_division=0),
        'recall':    recall_score(labels, preds, average='binary', zero_division=0),
    }

In [ ]:
# 10. Оқыту аргументтері
from transformers import TrainingArguments, Trainer

output_dir = f'{SAVE_DIR}/models/ruBert-base-disinfo'

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=WD,
    warmup_ratio=WARMUP,
    gradient_accumulation_steps=GRAD_ACCUM,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),   # FP16 тек GPU-да
    seed=42,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    compute_metrics=compute_metrics,
)

steps_per_epoch = len(tokenized['train']) // (BATCH_SIZE * GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
print(f'Эпохадағы қадамдар: {steps_per_epoch}')
print(f'Барлық қадамдар:    {total_steps}')
print(f'Оқытуды бастаймыз...')

In [ ]:
# 11. ОҚЫТУ — ~60-80 минут T4 GPU-да
import time

t0 = time.time()
train_result = trainer.train()
elapsed = time.time() - t0

print(f'\nОқыту аяқталды!')
print(f'Уақыт:  {elapsed/60:.1f} мин')
print(f'Loss:   {train_result.metrics["train_loss"]:.4f}')
print(f'Сэмпл/сек: {train_result.metrics["train_samples_per_second"]:.1f}')

## 4. Бағалау (Evaluation)

In [ ]:
# 12. Валидация + тест
print('=== Валидация ===')
val_metrics = trainer.evaluate(tokenized['validation'])
for k, v in val_metrics.items():
    if k.startswith('eval_'):
        print(f'  {k:<20s} {v:.4f}')

print('\n=== Тест ===')
test_metrics = trainer.evaluate(tokenized['test'])
for k, v in test_metrics.items():
    if k.startswith('eval_'):
        print(f'  {k:<20s} {v:.4f}')

test_f1 = test_metrics.get('eval_f1', 0)
test_acc = test_metrics.get('eval_accuracy', 0)
print(f'\n>>> Тест F1: {test_f1:.1%}  |  Accuracy: {test_acc:.1%}')
if test_f1 >= 0.90:
    print('Нәтиже: ЖАҚСЫ (F1 ≥ 90%)')
elif test_f1 >= 0.80:
    print('Нәтиже: ҚАНАҒАТТАНАРЛЫҚ (F1 ≥ 80%)')
else:
    print('ЕСКЕРТУ: F1 < 80% — деректерді немесе гиперпараметрлерді тексеріңіз')

In [ ]:
# 13. Confusion matrix + жіберілген мысалдар
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

pred_output = trainer.predict(tokenized['test'])
preds  = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['Сенімді', 'Дезинформация'],
    yticklabels=['Сенімді', 'Дезинформация'],
)
ax.set_xlabel('Болжам')
ax.set_ylabel('Шындық')
ax.set_title(f'Confusion Matrix (тест)  F1={test_f1:.1%}')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/models/confusion_matrix.png', dpi=100)
plt.show()

# Жіберілген мысалдар
test_texts = dataset['test']['text']
errors = [(test_texts[i], labels[i], preds[i])
          for i in range(len(labels)) if labels[i] != preds[i]]
print(f'\nЖіберілген: {len(errors)} / {len(labels)} ({len(errors)/len(labels):.1%})')
print('\nТоп-5 жіберілген мысал:')
for text, true, pred in errors[:5]:
    print(f'  Шындық={true} Болжам={pred}: {text[:100]}...')

In [ ]:
# 14. Үздік модельді сақтау
best_model_path = f'{SAVE_DIR}/models/best_model'
trainer.save_model(best_model_path)
tokenizer.save_pretrained(best_model_path)
print(f'Үздік модель сақталды: {best_model_path}')
print('Файлдар:')
for f in sorted(os.listdir(best_model_path)):
    size = os.path.getsize(f'{best_model_path}/{f}') / 1024**2
    print(f'  {f:<35s} {size:.1f} MB')

## 5. ONNX экспорт + INT8 квантизация

In [ ]:
# 15. ONNX-ке экспорт
from optimum.onnxruntime import ORTModelForSequenceClassification

onnx_dir = f'{SAVE_DIR}/models/onnx'
os.makedirs(onnx_dir, exist_ok=True)

print('ONNX-ке экспорттау...')
ort_model = ORTModelForSequenceClassification.from_pretrained(
    best_model_path, export=True
)
ort_model.save_pretrained(onnx_dir)
tokenizer.save_pretrained(onnx_dir)

onnx_path = f'{onnx_dir}/model.onnx'
onnx_mb = os.path.getsize(onnx_path) / 1024**2
print(f'ONNX модель: {onnx_mb:.1f} MB')

In [ ]:
# 16. INT8 квантизация (CPU инференс үшін)
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

quant_dir = f'{SAVE_DIR}/models/quant'
os.makedirs(quant_dir, exist_ok=True)

print('INT8 квантизация жүріп жатыр...')
quantizer = ORTQuantizer.from_pretrained(onnx_dir)
qconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)
quantizer.quantize(save_dir=quant_dir, quantization_config=qconfig)
tokenizer.save_pretrained(quant_dir)

# Квантизацияланған файл атын табу
quant_path = f'{quant_dir}/model_quantized.onnx'
if not os.path.exists(quant_path):
    quant_path = f'{quant_dir}/model.onnx'

quant_mb = os.path.getsize(quant_path) / 1024**2
print(f'Квантизацияланған модель: {quant_mb:.1f} MB')
print(f'Қысу коэффициенті: {onnx_mb/quant_mb:.1f}x  '
      f'({onnx_mb:.0f} MB → {quant_mb:.0f} MB)')

## 6. Инференс тесті

In [ ]:
# 17. ONNX инференс функциясы
import onnxruntime as ort

session = ort.InferenceSession(
    quant_path,
    providers=['CPUExecutionProvider'],
)
input_names = {inp.name for inp in session.get_inputs()}
print(f'ONNX inputs: {input_names}')

def predict_onnx(text: str) -> dict:
    enc = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length',
        return_tensors='np',
    )
    feed = {
        'input_ids':      enc['input_ids'].astype(np.int64),
        'attention_mask': enc['attention_mask'].astype(np.int64),
    }
    if 'token_type_ids' in input_names:
        feed['token_type_ids'] = enc.get(
            'token_type_ids', np.zeros_like(enc['input_ids'])
        ).astype(np.int64)

    logits = session.run(None, feed)[0][0]
    exp = np.exp(logits - logits.max())
    probs = exp / exp.sum()
    return {'reliable': float(probs[0]), 'disinformation': float(probs[1])}

In [ ]:
# 18. Жылдам тест
test_cases = [
    # Дезинформация (label=1) — болжам ДЕЗ болуы керек
    ('ДЕЗ', 'ШОК! Власти скрывают правду о ценах на продукты! Перешлите всем!!!'),
    ('ДЕЗ', 'СЕНСАЦИЯ! КНБ жасырын бақылау жүйесін іске қосты! Халық біліп қойсын!'),
    ('ДЕЗ', 'Акорда тайно готовит указ о повышении пенсионного возраста до 70 лет!'),
    ('ДЕЗ', 'ШҰҒЫЛ! Теңге бір айда 1000-ға дейін құлдырайды! Сақтаныңыз!'),
    # Сенімді (label=0) — болжам OK болуы керек
    ('OK ', 'По данным Нацбанка Казахстана, инфляция в марте составила 8.6% в годовом выражении.'),
    ('OK ', 'Мәжіліс мемлекеттік бюджет туралы заң жобасын үшінші оқылымда қабылдады.'),
    ('OK ', 'Согласно данным Бюро национальной статистики, ВВП вырос на 4.1% в 2024 году.'),
    ('OK ', 'ҚР Үкіметі инфляцияны төмендету бойынша іс-шаралар жоспарын бекітті.'),
]

print(f'{"Күтілген":<6} {"Болжам":<6} {"Дез%":<6}  Мәтін')
print('-' * 90)
correct = 0
for expected, text in test_cases:
    probs = predict_onnx(text)
    dez_pct = probs['disinformation']
    predicted = 'ДЕЗ' if dez_pct > 0.5 else 'OK '
    ok = '✓' if predicted.strip() == expected.strip() else '✗'
    if predicted.strip() == expected.strip():
        correct += 1
    print(f'{ok} {expected:<6} {predicted:<6} {dez_pct:.0%}    {text[:65]}...')

print(f'\nДұрыс: {correct}/{len(test_cases)} ({correct/len(test_cases):.0%})')

In [ ]:
# 19. Инференс жылдамдығын өлшеу
import time

WARMUP_RUNS = 5
BENCH_RUNS  = 50
bench_text  = 'Правительство Казахстана объявило о новых мерах поддержки малого бизнеса.'

for _ in range(WARMUP_RUNS):
    predict_onnx(bench_text)

times = []
for _ in range(BENCH_RUNS):
    t = time.perf_counter()
    predict_onnx(bench_text)
    times.append((time.perf_counter() - t) * 1000)

print(f'Инференс ({BENCH_RUNS} өлшеу):')
print(f'  Медиана: {np.median(times):.1f} мс')
print(f'  P95:     {np.percentile(times, 95):.1f} мс')
print(f'  Мин:     {min(times):.1f} мс')
print(f'  Макс:    {max(times):.1f} мс')

if np.median(times) < 100:
    print('  Инференс жылдамдығы ЖАҚСЫ (< 100 мс)')
else:
    print('  ЕСКЕРТУ: инференс баяу — CPU квантизациясын тексеріңіз')

## 7. Финалды сақтау (деплой үшін)

In [ ]:
# 20. Финалды файлдарды жинау
import shutil

final_dir = f'{SAVE_DIR}/models/final'
os.makedirs(f'{final_dir}/tokenizer', exist_ok=True)

# model.onnx → final/
shutil.copy2(quant_path, f'{final_dir}/model.onnx')

# tokenizer файлдары → final/tokenizer/
for fname in os.listdir(quant_dir):
    if fname.endswith(('.json', '.txt', '.model', '.vocab')):
        shutil.copy2(os.path.join(quant_dir, fname), f'{final_dir}/tokenizer/{fname}')

print(f'Финалды директория: {final_dir}')
print('Файлдар:')
for root, _, files in os.walk(final_dir):
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        rel   = os.path.relpath(fpath, final_dir)
        size  = os.path.getsize(fpath) / 1024**2
        print(f'  {rel:<40s} {size:.1f} MB')

In [ ]:
# 21. Нәтижелер қорытындысы
print('=' * 60)
print('  ОҚЫТУ НӘТИЖЕЛЕРІ')
print('=' * 60)
print(f'  Модель:     {MODEL_NAME}')
print(f'  Train сэмпл: {len(tokenized["train"])}')
print(f'  Эпохалар:   {EPOCHS}')
print()
print(f'  Тест F1:       {test_metrics.get("eval_f1", 0):.1%}')
print(f'  Тест Accuracy: {test_metrics.get("eval_accuracy", 0):.1%}')
print(f'  Тест Precision:{test_metrics.get("eval_precision", 0):.1%}')
print(f'  Тест Recall:   {test_metrics.get("eval_recall", 0):.1%}')
print()
print(f'  ONNX модель: {onnx_mb:.0f} MB → Квантизацияланған: {quant_mb:.0f} MB')
print(f'  Инференс:    {np.median(times):.0f} мс (медиана)')
print()
print('=' * 60)
print('  ДЕПЛОЙ НҰСҚАУЛЫҒЫ:')
print('  1. Google Drive-тан final/ папкасын жүктеңіз')
print('  2. model.onnx  → disinformation/models/model.onnx')
print('  3. tokenizer/* → disinformation/models/tokenizer/')
print('  4. uvicorn app.main:app --reload')
print('=' * 60)
print()
print('Дайын! Жергілікті тестілеу үшін FastAPI серверін іске қосыңыз.')